# Bitcoin Price Prediction with LSTM\n## EDA, Preprocessing, Model Training & Evaluation

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.preprocessing import MinMaxScaler\nfrom sklearn.metrics import mean_squared_error, mean_absolute_error\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import LSTM, Dense, Dropout\nplt.style.use('fivethirtyeight')\n%matplotlib inline

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('bitcoin.csv', parse_dates=['Date'], index_col='Date')\ndf = df.sort_index()\nprint(df.head())\nprint(df.info())\nprint(f'Dataset shape: {df.shape}')

In [ ]:
# Plot Close price\nplt.figure(figsize=(15,6))\nplt.plot(df.index, df['Close'])\nplt.title('Bitcoin Close Price Over Time')\nplt.xlabel('Date')\nplt.ylabel('Price (USD)')\nplt.show()

In [ ]:
# Correlation heatmap\nplt.figure(figsize=(10,8))\nsns.heatmap(df.corr(), annot=True, cmap='coolwarm')\nplt.title('Feature Correlation')\nplt.show()

## 2. Prepare Data for LSTM (use Close price)

In [ ]:
data = df[['Close']].values\nscaler = MinMaxScaler()\ndata_scaled = scaler.fit_transform(data)\n\ndef create_sequences(data, seq_length=60):\n    X, y = [], []\n    for i in range(seq_length, len(data)):\n        X.append(data[i-seq_length:i, 0])\n        y.append(data[i, 0])\n    return np.array(X), np.array(y)\n\nseq_length = 60\nX, y = create_sequences(data_scaled)\nX = X.reshape((X.shape[0], X.shape[1], 1))\n\nprint(f'X shape: {X.shape}, y shape: {y.shape}')

## 3. Train/Test Split

In [ ]:
split = int(0.8 * len(X))\nX_train, X_test = X[:split], X[split:]\ny_train, y_test = y[:split], y[split:]\nprint(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. Build & Train LSTM Model

In [ ]:
model = Sequential()\nmodel.add(LSTM(50, return_sequences=True, input_shape=(seq_length, 1)))\nmodel.add(Dropout(0.2))\nmodel.add(LSTM(50))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(1))\nmodel.compile(optimizer='adam', loss='mse')\n\nhistory = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=1)

In [ ]:
# Plot loss\nplt.figure(figsize=(12,4))\nplt.plot(history.history['loss'], label='Train Loss')\nplt.plot(history.history['val_loss'], label='Val Loss')\nplt.title('Model Loss')\nplt.legend()\nplt.show()

## 5. Predictions & Evaluation

In [ ]:
train_pred = model.predict(X_train)\ntest_pred = model.predict(X_test)\n\n# Inverse transform\ntrain_pred = scaler.inverse_transform(np.hstack((np.zeros((len(train_pred),1)), train_pred)))[:,1]\ntest_pred = scaler.inverse_transform(np.hstack((np.zeros((len(test_pred),1)), test_pred)))[:,1]\ny_train_inv = scaler.inverse_transform(np.hstack((np.zeros((len(y_train),1)), y_train.reshape(-1,1))))[:,1]\ny_test_inv = scaler.inverse_transform(np.hstack((np.zeros((len(y_test),1)), y_test.reshape(-1,1))))[:,1]\n\nrmse = np.sqrt(mean_squared_error(y_test_inv, test_pred))\nmae = mean_absolute_error(y_test_inv, test_pred)\nprint(f'Test RMSE: ${rmse:.2f}')\nprint(f'Test MAE: ${mae:.2f}')

In [ ]:
# Plot predictions\nplt.figure(figsize=(15,6))\ntrain_dates = df.index[seq_length:split+seq_length]\ntest_dates = df.index[split+seq_length:]\n\nplt.plot(train_dates, y_train_inv, label='Train Actual')\nplt.plot(train_dates, train_pred, label='Train Pred')\nplt.plot(test_dates, y_test_inv, label='Test Actual')\nplt.plot(test_dates, test_pred, label='Test Pred')\nplt.title('Bitcoin Close Price Prediction')\nplt.legend()\nplt.show()

## 6. Future Predictions (next 30 days)\n\nlast_sequence = data_scaled[-seq_length:].reshape((1, seq_length, 1))\nfuture_preds = []\nfor _ in range(30):\n    next_pred = model.predict(last_sequence)[0,0]\n    future_preds.append(next_pred)\n    last_sequence = np.roll(last_sequence, -1, axis=1)\n    last_sequence[0, -1, 0] = next_pred\n\nfuture_preds = scaler.inverse_transform(np.hstack((np.zeros((30,1)), np.array(future_preds).reshape(-1,1))))[:,1]\nprint('Next 30 days predictions:', future_preds)

In [ ]:
# Save model\nmodel.save('models/lstm_model.h5')\nprint('Model saved!')